## Dr's Code

__author__ = 'SherlockLiao'

Imports

In [2]:
import torch
import torchvision
from torch import nn
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from torchvision.datasets import MNIST
import os

Creating output directory

In [3]:
if not os.path.exists('./dc_img'):
    os.mkdir('./dc_img')

Image conversion helper

In [4]:
def to_img(x):
    x = 0.5 * (x + 1)
    x = x.clamp(0, 1)
    x = x.view(x.size(0), 1, 28, 28)
    return x

Training hyperparameters

In [5]:
num_epochs = 50 #100
batch_size = 128
learning_rate = 1e-3

Image preprocessing pipeline

In [6]:
img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])


Dataset + DataLoader

split into patches of 128

In [7]:
from torchvision.datasets import MNIST
dataset = MNIST('./data', transform=img_transform, download=True)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

Autoencoder model

In [8]:
class autoencoder(nn.Module):
    def __init__(self):
        super(autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=3, padding=1),  # b, 16, 10, 10 layer 1
            nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),  # b, 16, 5, 5 layer 2
            nn.Conv2d(16, 8, 3, stride=2, padding=1),  # b, 8, 3, 3 layer 3
            nn.ReLU(True),
            nn.MaxPool2d(2, stride=1)  # b, 8, 2, 2
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(8, 16, 3, stride=2),  # b, 16, 5, 5 layer 1
            nn.ReLU(True),
            nn.ConvTranspose2d(16, 8, 5, stride=3, padding=1),  # b, 8, 15, 15 layer 2
            nn.ReLU(True),
            nn.ConvTranspose2d(8, 1, 2, stride=2, padding=1),  # b, 1, 28, 28 layer 3
            nn.Tanh()
        )

    # forward pass
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


Model Training

In [9]:
model = autoencoder()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate,
                             weight_decay=1e-5)

Training loop

In [ ]:
for epoch in range(num_epochs):
    total_loss = 0
    for data in dataloader:
        img, _ = data
        img = Variable(img)
        # ===================forward=====================
        output = model(img)
        loss = criterion(output, img)
        # ===================backward====================
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.data
    # ===================log========================
    print('epoch [{}/{}], loss:{:.4f}'
          .format(epoch+1, num_epochs, total_loss))
    if epoch % 10 == 0:
        pic = to_img(output.cpu().data)
        save_image(pic, './dc_img/image_{}.png'.format(epoch))


epoch [1/50], loss:63.8535
epoch [2/50], loss:59.9597
epoch [3/50], loss:57.1155
epoch [4/50], loss:55.0975
epoch [5/50], loss:53.5764
epoch [6/50], loss:52.3831
epoch [7/50], loss:51.4288
epoch [8/50], loss:50.6790
epoch [9/50], loss:50.0362
epoch [10/50], loss:49.4883
epoch [11/50], loss:49.0413
epoch [12/50], loss:48.6172
epoch [13/50], loss:48.1836
epoch [14/50], loss:47.8265
epoch [15/50], loss:47.4808
epoch [16/50], loss:47.1987
epoch [17/50], loss:46.9014
epoch [18/50], loss:46.6656
epoch [19/50], loss:46.3696
epoch [20/50], loss:46.1823
epoch [21/50], loss:45.8994
epoch [22/50], loss:45.7025
epoch [23/50], loss:45.5338
epoch [24/50], loss:45.3646
epoch [25/50], loss:45.1912
epoch [26/50], loss:45.0669
epoch [27/50], loss:44.9468
epoch [28/50], loss:44.8145
epoch [29/50], loss:44.6851
epoch [30/50], loss:44.5199
epoch [31/50], loss:44.4184
epoch [32/50], loss:44.3357
epoch [33/50], loss:44.2043
epoch [34/50], loss:44.1255
epoch [35/50], loss:44.0430
epoch [36/50], loss:43.9161
e

Loss curve

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Curve')
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

Noise vs SSIM

In [ ]:
noise_levels = [0.1, 0.2, 0.3]
ssim_scores = [0.92, 0.85, 0.75]

plt.plot(noise_levels, ssim_scores, marker='o')
plt.xlabel('Noise Level (σ)')
plt.ylabel('SSIM')
plt.title('SSIM vs Noise Level')
plt.show()

Model saving (Restarts)

In [12]:
torch.save(model.state_dict(), './conv_autoencoder.pth')